# 02 — Data Readiness

**Owner:** Person A (modelling track).

**Rubric line:** Data quality, target definition, train/test split, data
dictionary, reproducibility.

Loads the raw CSV, applies safe pre-split cleaning, splits 80/20
stratified, and persists to `data/interim/` so downstream notebooks load
the same train/test pair.

In [1]:
# --- Setup --------------------------------------------------------------
# Make `src/` importable regardless of where the notebook is launched from.
import sys, pathlib
PROJECT_ROOT = pathlib.Path.cwd()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config, data

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

Matplotlib is building the font cache; this may take a moment.


## 2.1 — Load raw data

In [2]:
df_raw = data.load_raw()
print(f"Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head()

Shape: 41,188 rows x 21 columns


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## 2.2 — Profile: shape, dtypes, missing values, target balance

In [3]:
# Shape & memory
mem_mb = df_raw.memory_usage(deep=True).sum() / 1024**2
print(f"Shape  : {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(f"Memory : {mem_mb:.2f} MB\n")

# Dtypes
print("Column dtypes:")
print(df_raw.dtypes.to_string())
print()

# Null counts per column
null_counts = df_raw.isnull().sum()
print("Null counts per column:")
if null_counts.sum() == 0:
    print("  (no nulls in raw data)")
else:
    print(null_counts[null_counts > 0].to_string())
print()

# Target distribution: counts and proportions
target_counts = df_raw[config.TARGET_COL].value_counts()
target_props  = df_raw[config.TARGET_COL].value_counts(normalize=True)
target_summary = pd.DataFrame({"count": target_counts, "proportion": target_props})
print("Target distribution (counts and proportions):")
print(target_summary.to_string())

Shape  : 41,188 rows x 21 columns
Memory : 9.09 MB

Column dtypes:
age                 int64
job                   str
marital               str
education             str
default               str
housing               str
loan                  str
contact               str
month                 str
day_of_week           str
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome              str
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                     str

Null counts per column:
  (no nulls in raw data)

Target distribution (counts and proportions):
     count  proportion
y                     
no   36548    0.887346
yes   4640    0.112654


## 2.3 — Light cleaning (safe to do BEFORE the split)

- Encode target as 0/1.
- Replace `pdays == 999` sentinel with NaN, add `was_previously_contacted` flag.
- Strip whitespace from string cells.

**Note:** any imputation, scaling, or encoding that *learns from data* is
applied AFTER the split inside the modelling pipeline (see `src/features.py`).
We never fit on the full dataset.

In [4]:
df_clean = data.light_clean(df_raw)

# Verify: target is 0/1 integer
assert pd.api.types.is_integer_dtype(df_clean[config.TARGET_COL]), \
    f"Target dtype {df_clean[config.TARGET_COL].dtype} is not integer"
assert set(df_clean[config.TARGET_COL].unique()).issubset({0, 1}), \
    "Target contains values other than 0 and 1"
print(f"OK  target dtype: {df_clean[config.TARGET_COL].dtype}, "
      f"values: {sorted(df_clean[config.TARGET_COL].unique())}")

# Verify: pdays sentinel replaced with NaN
assert (df_clean["pdays"] == config.PDAYS_NOT_CONTACTED_SENTINEL).sum() == 0, \
    "pdays still contains sentinel value 999"
pdays_nan_pct = df_clean["pdays"].isna().mean()
print(f"OK  pdays sentinel removed; {pdays_nan_pct:.1%} of rows are NaN (never contacted)")

# Verify: was_previously_contacted flag added
assert "was_previously_contacted" in df_clean.columns, \
    "Missing 'was_previously_contacted' column"
vc = df_clean["was_previously_contacted"].value_counts().to_dict()
print(f"OK  was_previously_contacted added: {vc}")

# Verify: whitespace stripped from all string columns
obj_cols = df_clean.select_dtypes(include="object").columns
for c in obj_cols:
    has_ws = (df_clean[c].str.startswith(" ") | df_clean[c].str.endswith(" ")).sum()
    assert has_ws == 0, f"Column '{c}' still has leading/trailing whitespace"
print(f"OK  whitespace stripped across {len(obj_cols)} string columns")

df_clean.head()

OK  target dtype: int64, values: [np.int64(0), np.int64(1)]
OK  pdays sentinel removed; 96.3% of rows are NaN (never contacted)
OK  was_previously_contacted added: {0: 39673, 1: 1515}
OK  whitespace stripped across 10 string columns


C:\Users\JasperGeltenMalelion\AppData\Local\Temp\ipykernel_520\4064106435.py:24: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df_clean.select_dtypes(include="object").columns


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,was_previously_contacted
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,NaN,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,NaN,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,NaN,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,NaN,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
4,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,NaN,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0


## 2.4 — Train/test split (stratified 80/20)

In [5]:
train_df, test_df = data.split_train_test(df_clean)

overall_rate = df_clean[config.TARGET_COL].mean()
train_rate   = train_df[config.TARGET_COL].mean()
test_rate    = test_df[config.TARGET_COL].mean()

print(f"Overall : {len(df_clean):>6,} rows  positive rate: {overall_rate:.4%}")
print(f"Train   : {len(train_df):>6,} rows  positive rate: {train_rate:.4%}")
print(f"Test    : {len(test_df):>6,} rows  positive rate: {test_rate:.4%}")

# Verify: train vs test within 0.1 pp
diff_pp = abs(train_rate - test_rate) * 100
assert diff_pp <= 0.1, \
    f"Train/test positive-rate gap {diff_pp:.4f} pp exceeds 0.1 pp tolerance"
print(f"\nOK  |train_rate - test_rate| = {diff_pp:.4f} pp  (threshold: 0.1 pp)")

# Verify: both splits within 0.2 pp of the overall positive rate
for split_name, split_rate in [("Train", train_rate), ("Test", test_rate)]:
    gap_pp = abs(split_rate - overall_rate) * 100
    assert gap_pp <= 0.2, (
        f"{split_name} rate {split_rate:.4%} deviates {gap_pp:.4f} pp "
        f"from overall {overall_rate:.4%}"
    )
print(f"OK  both splits within 0.2 pp of overall rate ({overall_rate:.4%})")

Overall : 41,188 rows  positive rate: 11.2654%
Train   : 32,950 rows  positive rate: 11.2656%
Test    :  8,238 rows  positive rate: 11.2649%

OK  |train_rate - test_rate| = 0.0007 pp  (threshold: 0.1 pp)
OK  both splits within 0.2 pp of overall rate (11.2654%)


## 2.5 — Persist to data/interim/

In [6]:
data.save_interim(train_df, test_df)

train_path = config.INTERIM_DIR / "train.parquet"
test_path  = config.INTERIM_DIR / "test.parquet"

assert train_path.exists(), f"train.parquet not found at {train_path}"
assert test_path.exists(),  f"test.parquet not found at {test_path}"

train_kb = train_path.stat().st_size / 1024
test_kb  = test_path.stat().st_size / 1024
print(f"Saved  train.parquet  {train_kb:,.0f} KB  {len(train_df):,} rows")
print(f"Saved  test.parquet   {test_kb:,.0f} KB  {len(test_df):,} rows")

Saved  train.parquet  342 KB  32,950 rows
Saved  test.parquet   99 KB  8,238 rows


## 2.6 — Data dictionary (1-2 lines per field)

| Column | Type | Description |
|--------|------|-------------|
| age | numeric | Customer age in years. |
| job | categorical | Occupation category (12 levels including 'unknown'). |
| marital | categorical | Marital status (married/single/divorced/unknown). |
| education | categorical | Highest education level. |
| default | categorical | Has credit in default? (yes/no/unknown). |
| housing | categorical | Has housing loan? (yes/no/unknown). |
| loan | categorical | Has personal loan? (yes/no/unknown). |
| contact | categorical | Communication channel (cellular/telephone). |
| month | categorical | Month of last contact (jan-dec). |
| day_of_week | categorical | Day of last contact (mon-fri). |
| duration | numeric | **LEAKY.** Last call duration in seconds -- known only after the call. Excluded from the deployable model. |
| campaign | numeric | Number of contacts during this campaign for this customer. |
| pdays | numeric | Days since last contact in a previous campaign (NaN = never contacted). |
| previous | numeric | Number of contacts in previous campaigns. |
| poutcome | categorical | Outcome of the previous campaign (success/failure/nonexistent). |
| emp.var.rate | numeric | Employment variation rate (quarterly macro indicator). |
| cons.price.idx | numeric | Consumer price index (monthly macro). |
| cons.conf.idx | numeric | Consumer confidence index (monthly macro). |
| euribor3m | numeric | 3-month Euribor rate (daily macro). |
| nr.employed | numeric | Number of employees, in thousands (quarterly macro). |
| was_previously_contacted | derived | 1 if customer was contacted in a prior campaign, else 0. |
| **y** | **target** | **1 if the customer subscribed to the term deposit, else 0.** |

## 2.7 — Summary

| Item | Value |
|------|-------|
| Total rows (raw) | 41,188 |
| Train rows | ~32,950 (80%) |
| Test rows | ~8,238 (20%) |
| Positive rate (overall) | ~11.3% |
| Split strategy | Stratified on `y`, seed 42 |
| Interim files | `data/interim/train.parquet`, `data/interim/test.parquet` |

**Data quality observations:**

- **No missing values in the raw CSV.** The only sentinel requiring treatment is `pdays = 999`, which encodes "never previously contacted" (~96% of rows). This is replaced with NaN and a binary flag `was_previously_contacted` is added.
- **Class imbalance is mild (~11.3% positive).** Downstream models use `class_weight="balanced"` to handle this; no oversampling is applied at this stage.
- **`duration` is retained but flagged as leaky** -- it is excluded from the deployable feature set via `config.LEAKY_COLS`. It is only used for the leaky-benchmark contrast in notebook 04.
- **Macro-economic features** (`emp.var.rate`, `euribor3m`, `nr.employed`, etc.) will have very low within-period variance; they are kept as-is since they capture campaign timing effects.
- No imputation, scaling, or encoding has been applied here -- all such transformations happen inside `src/features.py` after the split to prevent data leakage.